### 4.4 Definição do Hardware

In [ ]:
# Verificar o status da GPU para garantir que o Colab alocou a placa de vídeo
!nvidia-smi

### 4.5 Configuração do Ambiente (Faster R-CNN e Dependências)

In [ ]:
import torch
import torchvision
print(f"PyTorch Versão: {torch.__version__} | Torchvision Versão: {torchvision.__version__}")

### Conexão com a Nuvem (Google Drive)

In [ ]:
from google.colab import drive
import os

# Montar o Google Drive
drive.mount('/content/drive')

# Desmontar o Google Drive
# drive.flush_and_unmount()

### 4.3 Pré-processamento dos Datasets

In [ ]:
import os
import shutil
import torchvision
from torchvision.transforms import transforms as T
import torch

# ==============================================================================
# 1. CAMINHOS E DESCOMPACTAÇÃO DIRETA NA VM
# ==============================================================================
caminho_zip_drive = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/FasterRCNN_dataset.zip"
pasta_local_vm = "/content/FasterRCNN_dataset"
zip_local_vm = "/content/FasterRCNN_dataset.zip"

print("🔍 Verificando arquivo no Drive...")
if not os.path.exists(caminho_zip_drive):
    raise FileNotFoundError(f"❌ Arquivo não encontrado no Drive: {caminho_zip_drive}")

# Limpar tentativas e resíduos anteriores na VM local
if os.path.exists(zip_local_vm):
    os.remove(zip_local_vm)
if os.path.exists(pasta_local_vm):
    shutil.rmtree(pasta_local_vm)

os.makedirs(pasta_local_vm, exist_ok=True)

print("🚚 Copiando o FasterRCNN_dataset.zip do Drive para o SSD local da VM...")
shutil.copy2(caminho_zip_drive, zip_local_vm)
print(f"📦 Arquivo copiado com sucesso ({os.path.getsize(zip_local_vm) / (1024**3):.2f} GB).")

print("⚡ Descompactando diretamente na VM usando o utilitário nativo...")
# Usamos o unzip nativo do Linux do Colab que é muito mais performático para ficheiros grandes
!unzip -q {zip_local_vm} -d {pasta_local_vm}

# Remove o zip local para libertar espaço em disco na VM, mantendo apenas a pasta extraída
if os.path.exists(zip_local_vm):
    os.remove(zip_local_vm)
print("✅ Descompactação concluída na VM!")


# ==============================================================================
# 2. DEFINIÇÃO DO DATASET (COCO FORMAT) E DATALOADERS
# ==============================================================================

# Classe Customizada utilizando o CocoDetection nativo do PyTorch
class FasterRCNNCocoDataset(torchvision.datasets.CocoDetection):
    def __init__(self, root, annFile, transforms=None):
        super(FasterRCNNCocoDataset, self).__init__(root, annFile, transforms=None)
        self.meus_transforms = transforms
        
        # 🌟 CORREÇÃO 1: Mapear qualquer ID que venha do JSON para uma ordem estrita 1, 2, 3...
        # Isso impede que o modelo quebre se o seu dataset tiver ID 0 ou números pulados.
        ids_originais = sorted(self.coco.getCatIds())
        self.mapa_seguro_classes = {id_original: idx + 1 for idx, id_original in enumerate(ids_originais)}

    def __getitem__(self, idx):
        img, target_orig = super(FasterRCNNCocoDataset, self).__getitem__(idx)
        largura_orig, altura_orig = img.size
        
        boxes = []
        labels = []
        
        for ann in target_orig:
            bbox = ann['bbox']
            
            # Regra de Três para redimensionar as coordenadas
            x_min = (bbox[0] / largura_orig) * 640
            y_min = (bbox[1] / altura_orig) * 640
            x_max = ((bbox[0] + bbox[2]) / largura_orig) * 640
            y_max = ((bbox[1] + bbox[3]) / altura_orig) * 640
            
            # 🌟 CORREÇÃO 2: "Clipping" (Corte). Garante que não passe de 640 e nem seja menor que 0
            x_min = max(0.0, min(x_min, 640.0))
            y_min = max(0.0, min(y_min, 640.0))
            x_max = max(0.0, min(x_max, 640.0))
            y_max = max(0.0, min(y_max, 640.0))
            
            # Evita caixas com área zero após o corte
            if (x_max - x_min) > 1.0 and (y_max - y_min) > 1.0:
                boxes.append([x_min, y_min, x_max, y_max])
                # Pega o ID original e converte no nosso ID seguro (1, 2, 3...)
                id_seguro = self.mapa_seguro_classes[ann['category_id']]
                labels.append(id_seguro)
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        
        if self.meus_transforms:
            img = self.meus_transforms(img)
            
        return img, target

# Transformação
dataset_transforms = T.Compose([
    T.Resize((640, 640)), 
    T.ToTensor(),
])

# Instanciação
train_dataset = FasterRCNNCocoDataset(os.path.join(pasta_local_vm, "images/train"), os.path.join(pasta_local_vm, "annotations/train.json"), transforms=dataset_transforms)
val_dataset = FasterRCNNCocoDataset(os.path.join(pasta_local_vm, "images/val"), os.path.join(pasta_local_vm, "annotations/val.json"), transforms=dataset_transforms)
test_dataset = FasterRCNNCocoDataset(os.path.join(pasta_local_vm, "images/test"), os.path.join(pasta_local_vm, "annotations/test.json"), transforms=dataset_transforms)

def collate_fn(batch):
    return tuple(zip(*batch))

# DataLoaders com batch size 2
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4)

print("=" * 50)
print(f"📊 Amostras de Treino: {len(train_dataset)} imagens")
print(f"📊 Amostras de Validação: {len(val_dataset)} imagens")
print(f"📊 Amostras de Teste: {len(test_dataset)} imagens")
print("✅ Dataset carregado com validação rigorosa de Classes e Bounding Boxes!")
print("=" * 50)

### 4.6 Definição de Hiperparâmetros e 4.7 Treinamento do Modelo

In [ ]:
import os
import time  # <-- Importação adicionada
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from datetime import datetime
from tqdm import tqdm

# 1. Criação do diretório de saída estruturado no seu Drive
agora = datetime.now()
nome_treino_rcnn = agora.strftime("Treino_faster_rcnn_%Y-%m-%d_%Hh%M")
diretorio_drive_rcnn = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_FasterRCNN"
os.makedirs(f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights", exist_ok=True)

print(f"📁 Destino dos resultados: {diretorio_drive_rcnn}/{nome_treino_rcnn}")

# Mapear a quantidade correta de classes
num_classes = len(train_dataset.mapa_seguro_classes) + 1 
print(f"📌 Número de classes detectadas para o modelo: {num_classes} (incluindo background)")

# 2. Inicializar o Faster R-CNN com Pesos Pré-treinados (Transfer Learning)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

# Substituir o cabeçalho de classificação para adequar ao número de classes do seu TCC
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

# 3. Otimizador padrão de Two-Stage Detectors
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# --- NOVAS VARIÁVEIS DO EARLY STOPPING ---
epochs = 100             # Limite alto, o modelo vai parar sozinho antes disso
patience = 10            # Tolerância: quantas épocas esperar sem melhoria
epochs_no_improve = 0    # Contador de épocas frustradas
best_val_loss = float('inf') 

# --- MARCAÇÃO DE INÍCIO DO TREINAMENTO TOTAL ---
tempo_inicio_total = time.time()

# 4. Loop de Treinamento Principal
for epoch in range(epochs):
    # --- Início do timer da época ---
    tempo_inicio_epoca = time.time()
    
    # ==========================================
    # FASE 1: TREINO
    # ==========================================
    model.train()
    epoch_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for images, targets in progress_bar:
        # CORREÇÃO: Converter a tupla de imagens do DataLoader explicitamente para a GPU
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Ignorar amostras sem anotações para não quebrar a Region Proposal Network (RPN)
        if any(len(t["boxes"]) == 0 for t in targets):
            continue

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_train_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())
        
    media_train_loss = epoch_train_loss / len(train_loader)
    
    # ==========================================
    # FASE 2: VALIDAÇÃO (O Motor do Early Stopping)
    # ==========================================
    epoch_val_loss = 0
    
    with torch.no_grad(): # Não calcula gradientes na validação
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
        
        for images, targets in val_bar:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # ATENÇÃO: O Faster R-CNN também precisa ficar em .train() 
            # para devolver o dicionário de loss em vez das previsões finais.
            model.train() 
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            epoch_val_loss += losses.item()

    media_val_loss = epoch_val_loss / len(val_loader)
    
    # ==========================================
    # FASE 3: LÓGICA DE PACIÊNCIA E SALVAMENTO
    # ==========================================
    tempo_fim_epoca = time.time()
    minutos_ep, segundos_ep = divmod(tempo_fim_epoca - tempo_inicio_epoca, 60)
    
    print(f"📉 Fim da Epoch {epoch+1} - Train Loss: {media_train_loss:.4f} | Val Loss: {media_val_loss:.4f}")
    print(f"⏱️ Tempo: {int(minutos_ep)}m {int(segundos_ep)}s")
    
    # Salvamento Periódico Automatizado (last.pt)
    torch.save(model.state_dict(), f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/last.pt")
    
    # A lógica principal de Paciência baseada na VALIDAÇÃO:
    if media_val_loss < best_val_loss:
        # Se melhorou, salva o modelo e zera o contador de paciência
        best_val_loss = media_val_loss
        epochs_no_improve = 0
        melhor_epoca = epoch + 1 # <--- Salva a melhor época para o relatório futuro
        torch.save(model.state_dict(), f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/best.pt")
        print("🏆 Excelente! Melhor modelo atualizado (Baseado na Val Loss) e salvo no Drive.")
    else:
        # Se piorou, aumenta o contador
        epochs_no_improve += 1
        print(f"⚠️ Sem melhora na Val Loss há {epochs_no_improve} época(s).")
        
        # Se o contador atingir a paciência, interrompe o treino
        if epochs_no_improve >= patience:
            print("-" * 50)
            print(f"🛑 EARLY STOPPING ACIONADO! O modelo parou de generalizar.")
            print(f"O melhor peso salvo está em 'best.pt' (da época {melhor_epoca}).")
            break # Encerra o loop das épocas
            
    print("-" * 50)

# --- Fim do timer total e cálculo ---
tempo_fim_total = time.time()
duracao_total = tempo_fim_total - tempo_inicio_total
minutos_tot, segundos_tot = divmod(duracao_total, 60)

print("\n" + "=" * 60)
print("✅ Treinamento do Faster R-CNN concluído!")
print(f"⏱️ Tempo Total do Treinamento: {int(minutos_tot)}m {int(segundos_tot)}s")
print("=" * 60)

### 4.7 Inferência Prática e Avaliação de Métricas (Conjunto de Testes)

In [ ]:
!pip install torchmetrics -U

import time
import torch
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision


def avaliar_metricas_rcnn(model, data_loader, device):
    model.eval()

    # Métricas COCO
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

    tempos_inferencia = []

    # Contadores para Precision e Recall
    tp = 0
    fp = 0
    fn = 0

    IOU_THRESHOLD = 0.5
    SCORE_THRESHOLD = 0.5

    with torch.no_grad():

        for images, targets in data_loader:

            images = [img.to(device) for img in images]

            # ---------------- Inferência ----------------
            start_time = time.time()
            outputs = model(images)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end_time = time.time()

            tempos_inferencia.append(
                (end_time - start_time) / len(images)
            )

            # ------------ Atualiza mAP ------------
            preds = []
            for output in outputs:
                preds.append({
                    "boxes": output["boxes"].cpu(),
                    "scores": output["scores"].cpu(),
                    "labels": output["labels"].cpu()
                })

            gts = []
            for t in targets:
                gts.append({
                    "boxes": t["boxes"].cpu(),
                    "labels": t["labels"].cpu()
                })

            metric.update(preds, gts)

            # ----------- Calcula Precision/Recall ------------
            for output, target in zip(outputs, targets):

                pred_boxes = output["boxes"].cpu()
                pred_scores = output["scores"].cpu()
                pred_labels = output["labels"].cpu() # ADICIONADO


                gt_boxes = target["boxes"].cpu()
                gt_labels = target["labels"].cpu() # ADICIONADO

                # Remove previsões de baixa confiança
                keep = pred_scores >= SCORE_THRESHOLD
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep] # ADICIONADO

                matched_gt = set()

                for pred_box, pred_label in zip(pred_boxes, pred_labels):

                    melhor_iou = 0
                    melhor_gt = -1

                    for i, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):

                        if i in matched_gt:
                            continue

                        if pred_label != gt_label:
                            continue

                        iou = box_iou(
                            pred_box.unsqueeze(0),
                            gt_box.unsqueeze(0)
                        )[0, 0].item()

                        if iou > melhor_iou:
                            melhor_iou = iou
                            melhor_gt = i

                    if melhor_iou >= IOU_THRESHOLD:
                        tp += 1
                        matched_gt.add(melhor_gt)
                    else:
                        fp += 1

                fn += len(gt_boxes) - len(matched_gt)

    resultados = metric.compute()

    tempo_medio = sum(tempos_inferencia) / len(tempos_inferencia)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    return resultados, tempo_medio, precision, recall


# ============================================================
# Carrega os pesos
# ============================================================

model.load_state_dict(
    torch.load(
        f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/best.pt"
    )
)

print("Executando inferência e calculando métricas (isso pode demorar um pouco)...")

resultados_map, tempo_medio, precision, recall = avaliar_metricas_rcnn(
    model,
    test_loader,
    device
)

# ============================================================
# Conversão para %
# ============================================================

precision *= 100
recall *= 100

map_50_95 = resultados_map["map"].item() * 100
map_50 = resultados_map["map_50"].item() * 100
mar_100 = resultados_map["mar_100"].item() * 100

tempo_medio_ms = tempo_medio * 1000

print("=" * 70)
print("📊 Avaliação Final no Conjunto de Testes - Faster R-CNN")
try:
    print(f"📌 Melhor peso carregado da Época: {melhor_epoca}")
except NameError:
    pass
print(f"⏱️ Tempo Médio:                 {tempo_medio_ms:.2f} ms/imagem")
print("-" * 70)
print("💡 Precision: TP / (TP + FP)")
print("💡 Recall: TP / (TP + FN)")
print("-" * 70)
print(f"🎯 Precisão:                    {precision:.2f}%")
print(f"🎯 Recall:                      {recall:.2f}%")
print(f"🎯 Recall Médio (mAR@100):      {mar_100:.2f}%")
print(f"🎯 mAP@0.5:                     {map_50:.2f}%")
print(f"🎯 mAP@0.5:0.95:                {map_50_95:.2f}%")
print("=" * 70)

# ============================================================
# Salva no Drive
# ============================================================

with open(
    f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/resultado_teste.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write("Métricas de Avaliação Final - Faster R-CNN (ResNet50 v2)\n")
    f.write(f"Data e Hora: {agora.strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # --- CORREÇÃO DAS ÉPOCAS E EARLY STOPPING ---
    try:
        f.write(f"Melhor peso encontrado na época: {melhor_epoca}\n")
        f.write(f"Treinamento interrompido na época: {epoch + 1} (Limite era {epochs})\n")
    except NameError:
        f.write(f"Número Limite de Épocas Configurado: {epochs}\n")

    f.write(f"Tempo Total de Treinamento: {int(minutos_tot / 60)}h {int(minutos_tot % 60)}m {int(segundos_tot % 60)}s\n")
    f.write(f"Tempo Total de Treinamento em minutos: {minutos_tot:.2f} min\n")
    f.write("-" * 60 + "\n\n")

    f.write("Precisão = TP / (TP + FP)\n")
    f.write("Recall = TP / (TP + FN)\n")
    f.write("mAP = Média da Average Precision (COCO)\n\n")

    f.write(f"Tempo médio por imagem: {tempo_medio_ms:.2f} ms\n")
    f.write(f"Precisão:               {precision:.2f}%\n")
    f.write(f"Recall:                 {recall:.2f}%\n")
    f.write(f"Recall Médio (mAR@100): {mar_100:.2f}%\n")
    f.write(f"mAP@0.5:                {map_50:.2f}%\n")
    f.write(f"mAP@0.5:0.95:           {map_50_95:.2f}%\n")

print("💾 Arquivo contendo as métricas gerado com sucesso no Drive!")

### Backup do teste

In [ ]:
import os

# Criar a pasta de resultados visuais do teste no Google Drive
pasta_teste_drive = f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/Teste_Final"
os.makedirs(pasta_teste_drive, exist_ok=True)

print(f"💾 Iniciando backup das métricas de teste para o Google Drive...")

# Salva um relatório de texto simples atualizado com as novas métricas
with open(f"{pasta_teste_drive}/relatorio_metricas.txt", "w", encoding="utf-8") as f:
    f.write("Métricas de Avaliação Final - Faster R-CNN (ResNet50 v2)\n")
    f.write(f"Data e Hora: {agora.strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # --- CORREÇÃO DAS ÉPOCAS E EARLY STOPPING ---
    try:
        f.write(f"Melhor peso encontrado na época: {melhor_epoca}\n")
        f.write(f"Treinamento interrompido na época: {epoch + 1} (Limite era {epochs})\n")
    except NameError:
        f.write(f"Número Limite de Épocas Configurado: {epochs}\n")
        
    f.write(f"Tempo Total de Treinamento: {int(minutos_tot / 60)}h {int(minutos_tot % 60)}m {int(segundos_tot % 60)}s\n")
    f.write(f"Tempo Total de Treinamento em minutos: {minutos_tot:.2f} min\n")
    f.write("-" * 60 + "\n")
    f.write("Precisão = TP / (TP + FP)\n")
    f.write("Recall = TP / (TP + FN)\n")
    f.write("mAP = Média da Average Precision (COCO)\n")
    f.write("-" * 60 + "\n\n")
    f.write(f"Tempo Médio:            {tempo_medio_ms:.2f} ms / imagem\n")
    f.write(f"Precisão:               {precision:.2f}%\n")
    f.write(f"Recall:                 {recall:.2f}%\n")
    f.write(f"Recall Médio (mAR@100): {mar_100:.2f}%\n")
    f.write(f"mAP@0.5:                {map_50:.2f}%\n")
    f.write(f"mAP@0.5:0.95:           {map_50_95:.2f}%\n")

print(f"✅ Backup concluído com sucesso!")
print(f"👉 Salvo em: {pasta_teste_drive}")

### 4.8 Coleta De metricas visuais

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes
import cv2
import torch

# Dicionário reverso para exibir o nome da classe (busca do dataset COCO)
safe_to_orig = {v: k for k, v in train_dataset.mapa_seguro_classes.items()}
id_para_nome = {safe_id: train_dataset.coco.cats[orig_id]['name'] for safe_id, orig_id in safe_to_orig.items()}

def visualizar_e_salvar_amostras(model, dataset, device, pasta_destino, num_amostras=3, limiar_confianca=0.5):
    model.eval()
    
    print("\n>>> INFERÊNCIA PRÁTICA (Original vs Ground Truth vs Predições)")
    
    for i in range(num_amostras):
        # Pegar imagem diretamente do dataset para garantir a escala correta (640x640)
        img_tensor, target = dataset[i]
        
        with torch.no_grad():
            # O modelo espera uma lista de tensores
            output = model([img_tensor.to(device)])[0]
            
        # Converter imagem de float [0.0, 1.0] para uint8 [0, 255] exigido pelo draw_bounding_boxes
        img_uint8 = (img_tensor * 255).to(torch.uint8)
        
        # --- 1. PREPARAR ANOTAÇÕES REAIS (GROUND TRUTH) ---
        boxes_gt = target["boxes"]
        if len(boxes_gt) > 0:
            labels_gt = [id_para_nome.get(l.item(), f"ID {l.item()}") for l in target["labels"]]
            img_gt = draw_bounding_boxes(img_uint8, boxes_gt, labels=labels_gt, colors="green", width=3)
        else:
            img_gt = img_uint8
            
        # --- 2. PREPARAR PREDIÇÕES DA IA ---
        pred_scores = output["scores"]
        
        # Lógica de fallback: se nada superar o limiar, reduz para 15% para visualizar o que a IA tentou
        limiar_atual = limiar_confianca
        if len(pred_scores) > 0:
            max_score = pred_scores.max().item()
            print(f"ℹ️ Amostra {i} - Maior score de confiança predito: {max_score*100:.1f}%")
            if (pred_scores > limiar_atual).sum() == 0:
                limiar_atual = 0.15
                print(f"⚠️ Nenhuma predição superou {limiar_confianca*100:.0f}%. Reduzindo limiar para {limiar_atual*100:.0f}% na visualização.")
        else:
            print(f"ℹ️ Amostra {i} - Nenhuma predição gerada pelo modelo.")
            
        keep_idx = pred_scores > limiar_atual
        boxes_pred = output["boxes"][keep_idx].cpu()
        scores_pred = pred_scores[keep_idx].cpu()
        labels_pred_ids = output["labels"][keep_idx].cpu()
        
        if len(boxes_pred) > 0:
            labels_pred = [
                f"{id_para_nome.get(l.item(), f'ID {l.item()}')} {s.item()*100:.0f}%" 
                for l, s in zip(labels_pred_ids, scores_pred)
            ]
            img_pred = draw_bounding_boxes(img_uint8, boxes_pred, labels=labels_pred, colors="red", width=3)
        else:
            img_pred = img_uint8
            
        # --- 3. PLOTAR LADO A LADO E SALVAR NO DRIVE ---
        # ALTERAÇÃO AQUI: Mudamos de (1, 2) para (1, 3) e aumentamos a largura (figsize)
        fig, ax = plt.subplots(1, 3, figsize=(20, 7))
        
        # Coluna 1: Imagem Original (sem marcação)
        ax[0].imshow(img_uint8.permute(1, 2, 0).numpy())
        ax[0].set_title(f"Amostra {i} - Original", fontsize=14, fontweight='bold')
        ax[0].axis("off")

        # Coluna 2: Anotações Reais (Ground Truth)
        ax[1].imshow(img_gt.permute(1, 2, 0).numpy())
        ax[1].set_title(f"Amostra {i} - Anotações Reais", fontsize=14, fontweight='bold')
        ax[1].axis("off")
        
        # Coluna 3: Predição da IA
        ax[2].imshow(img_pred.permute(1, 2, 0).numpy())
        ax[2].set_title(f"Amostra {i} - Predição da IA (> {limiar_atual*100:.0f}%)", fontsize=14, fontweight='bold')
        ax[2].axis("off")
        
        plt.tight_layout()
        
        # Salvar a figura combinada diretamente no Drive (em RGB correto)
        caminho_salvar = f"{pasta_destino}/amostra_{i}_comparacao.jpg"
        plt.savefig(caminho_salvar, bbox_inches='tight', dpi=150)
        
        # Exibir no Colab
        plt.show()

print(f"📊 Gerando visualizações de inferência...")
print("-" * 60)

# Chama a função para processar, visualizar e salvar as 3 primeiras imagens do dataset de teste
visualizar_e_salvar_amostras(model, test_dataset, device, pasta_teste_drive, num_amostras=3, limiar_confianca=0.5)

print("\n✅ Fluxo finalizado! O modelo está treinado, avaliado e as comparações visuais estão salvas no Drive.")